In [1]:
import pandas as pd
import os
import numpy as np

C:\Users\marie\AppData\Local\Temp\ipykernel_9448\3363072451.py:1: DeprecationWarning: 
Pyarrow will become a required dependency of pandas in the next major release of pandas (pandas 3.0),
(to allow more performant data types, such as the Arrow string type, and better interoperability with other libraries)
but was not found to be installed on your system.
If this would cause problems for you,
please provide us feedback at https://github.com/pandas-dev/pandas/issues/54466
        
  import pandas as pd


In [2]:
input_filepath = r"data\feature_table.csv"
output_path = r"data"
mising_path  = r"C:\Users\marie\rep_codes\udder_project\delpro_vms\data\missing_teats.csv"

In [3]:
dtypes = {'cow':'str', 'frame':'Int64', 'filename':'str', 'udder_vol': np.float64, 'lf_vol':np.float64, 'rf_vol':np.float64, 'lb_vol':np.float64, \
 'rb_vol':np.float64, 'udder_sarea':np.float64, 'lf_sarea':np.float64, 'rf_sarea':np.float64, 'lb_sarea':np.float64, 'rb_sarea':np.float64, \
 'lf_angle':np.float64, 'rf_angle':np.float64, 'lb_angle':np.float64, 'rb_angle':np.float64, 'front_eu':np.float64, 'back_eu':np.float64, \
 'right_eu':np.float64, 'left_eu':np.float64, 'front_gd':np.float64, 'back_gd':np.float64, 'right_gd':np.float64, 'left_gd':np.float64,\
 'udder_peri':np.float64, 'udder_area':np.float64, 'udder_circ':np.float64, 'udder_exc':np.float64, 'lf_peri':np.float64, 'lf_area':np.float64, \
 'lf_circ':np.float64, 'lf_exc':np.float64, 'rf_peri':np.float64, 'rf_area':np.float64, 'rf_circ':np.float64, 'rf_exc':np.float64, 'lb_peri':np.float64, \
 'lb_area':np.float64, 'lb_circ':np.float64, 'lb_exc':np.float64, 'rb_peri':np.float64, 'rb_area':np.float64, 'rb_circ':np.float64, 'rb_exc':np.float64, \
 'lf_len':np.float64, 'rf_len':np.float64, 'lb_len':np.float64, 'rb_len':np.float64}

In [4]:
fdf = pd.read_csv(input_filepath)
fdf[fdf.columns[3:]] = fdf[fdf.columns[3:]].apply(pd.to_numeric, errors='coerce')

mt_df = pd.read_csv(mising_path)
cow_list = np.unique(mt_df.cow)
len(cow_list)

C:\Users\marie\AppData\Local\Temp\ipykernel_9448\3536355538.py:1: DtypeWarning: Columns (17,18,19,20) have mixed types. Specify dtype option on import or set low_memory=False.
  fdf = pd.read_csv(input_filepath)


30

In [5]:
# cange values of missing teats to missing
for cow in cow_list:
    teats_missing = mt_df[mt_df.cow == cow]["teat"].values
    for teat in teats_missing:
        columns_with_teat = [col for col in fdf.columns if teat in col]
        print(columns_with_teat)
        fdf.loc[fdf.cow == cow, columns_with_teat] = np.nan
        temp = fdf[fdf.cow == cow][columns_with_teat]
        # print(temp)
fdf.to_csv(os.path.join(output_path, "mfeature_table.csv"), index = False)

['rf_vol', 'rf_sarea', 'rf_peri', 'rf_area', 'rf_circ', 'rf_exc', 'rf_len']
['lf_vol', 'lf_sarea', 'lf_peri', 'lf_area', 'lf_circ', 'lf_exc', 'lf_len']
['rf_vol', 'rf_sarea', 'rf_peri', 'rf_area', 'rf_circ', 'rf_exc', 'rf_len']
['lf_vol', 'lf_sarea', 'lf_peri', 'lf_area', 'lf_circ', 'lf_exc', 'lf_len']
['rf_vol', 'rf_sarea', 'rf_peri', 'rf_area', 'rf_circ', 'rf_exc', 'rf_len']
['rf_vol', 'rf_sarea', 'rf_peri', 'rf_area', 'rf_circ', 'rf_exc', 'rf_len']
['lf_vol', 'lf_sarea', 'lf_peri', 'lf_area', 'lf_circ', 'lf_exc', 'lf_len']
['rb_vol', 'rb_sarea', 'rb_peri', 'rb_area', 'rb_circ', 'rb_exc', 'rb_len']
['lb_vol', 'lb_sarea', 'lb_peri', 'lb_area', 'lb_circ', 'lb_exc', 'lb_len']
['rb_vol', 'rb_sarea', 'rb_peri', 'rb_area', 'rb_circ', 'rb_exc', 'rb_len']
['lb_vol', 'lb_sarea', 'lb_peri', 'lb_area', 'lb_circ', 'lb_exc', 'lb_len']
['lf_vol', 'lf_sarea', 'lf_peri', 'lf_area', 'lf_circ', 'lf_exc', 'lf_len']
['lb_vol', 'lb_sarea', 'lb_peri', 'lb_area', 'lb_circ', 'lb_exc', 'lb_len']
['lb_vol', '

In [6]:
len(np.unique(fdf.cow))

138

In [7]:
fdfg = fdf.drop(["frame", "filename"], axis = 1).groupby(["cow"]).median()

fdfg_cols = fdfg.columns
for colname in fdfg_cols:
    # print(colname)
    Q1 = np.nanpercentile(fdfg[colname], 25, method='midpoint')
    Q3 = np.nanpercentile(fdfg[colname], 75, method='midpoint')
    med = np.nanpercentile(fdfg[colname], 50, method='midpoint')
    IQR = Q3 - Q1
    low_thres = med - 1.5*IQR
    up_thres = med + 1.5*IQR
    out = (fdfg[colname] < low_thres)| (fdfg[colname] > up_thres)
    fdfg.loc[out, colname] = np.nan

fdfg = fdfg.reset_index()
fdfg.to_csv(os.path.join(output_path, "gmfeature_table.csv"), index = False)

In [8]:
fdf[fdf.columns[3:]] = fdf[fdf.columns[3:]].apply(pd.to_numeric, errors='coerce')
fdfg = fdf.drop(["frame", "filename"], axis = 1).groupby(["cow"]).median()
fdfg2 = fdfg.copy()
fdfg_cols = fdfg.columns

for colname in fdfg_cols:
    # print(colname)
    Q1 = np.nanpercentile(fdfg[colname], 25, method='midpoint')
    Q3 = np.nanpercentile(fdfg[colname], 75, method='midpoint')
    med = np.nanpercentile(fdfg[colname], 50, method='midpoint')
    IQR = Q3 - Q1
    low_thres = med - 1.5*IQR
    up_thres = med + 1.5*IQR
    out = (fdfg[colname] < low_thres)| (fdfg[colname] > up_thres)
    fdfg.loc[out, colname] = np.nan
    fdfg2.loc[out, colname] = np.nanmean(fdfg[colname])
fdfg = fdfg.reset_index()
fdfg.to_csv(os.path.join(output_path, "gmfeature_table.csv"), index = False)


In [9]:
cow_list2 = list(set(cow_list).intersection(set(fdfg2.index)))
fdfg2["missing"] = 0
fdfg2.loc[cow_list2, "missing"] = 1

fdfg2.to_csv(os.path.join(output_path, "gmfeature_table_cluster.csv"), index = True)

In [10]:
fdfg

,cow,udder_vol,lf_vol,rf_vol,lb_vol,rb_vol,udder_sarea,lf_sarea,rf_sarea,lb_sarea,...,lb_circ,lb_exc,rb_peri,rb_area,rb_circ,rb_exc,lf_len,rf_len,lb_len,rb_len
0,627,6.742235,2.085106,2.320265,0.655287,0.907118,2202.840957,676.526869,710.606157,388.758950,...,0.753654,0.540129,NaN,NaN,0.718000,0.600890,37.991288,41.337529,42.078602,36.358942
1,723,5.866114,3.189081,1.988038,0.497105,0.565673,1557.547077,620.012292,569.962040,259.224542,...,0.603719,0.842961,477.782792,8966.5,NaN,0.815459,46.126468,47.956094,32.716431,NaN
2,729,NaN,5.068295,4.355240,0.786221,0.891771,NaN,NaN,998.056447,255.710493,...,0.670033,0.571781,422.107648,10422.0,0.740037,0.664880,NaN,60.757677,48.859144,37.343963
3,736,8.529069,5.034365,1.684315,1.298735,0.676856,2190.923732,1036.329523,415.856293,525.530135,...,0.719037,0.549009,445.314755,9557.0,0.605616,NaN,NaN,NaN,51.872415,49.515135
4,764,6.316109,1.788540,2.241100,0.752663,0.685146,NaN,825.074567,901.399146,567.414139,...,0.644892,0.624028,452.268073,9417.0,0.581946,0.735450,43.875716,44.013406,37.066922,37.681559
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
133,1486,5.378144,1.734115,1.861548,0.768159,0.702433,1964.898450,521.314016,589.685346,475.684154,...,NaN,0.785654,496.498521,11302.0,0.572537,0.778734,37.961439,37.554606,27.339605,23.264238
134,1487,8.376260,5.059448,3.538225,0.712471,0.503287,1788.671257,869.119892,720.359164,208.901041,...,0.676632,0.791993,354.205627,5754.5,0.602927,0.738296,42.302186,51.276823,26.069107,NaN
135,1495,5.783034,1.668636,3.162655,0.557299,0.537761,1910.238056,434.406296,775.294209,346.581440,...,NaN,0.727476,514.640656,10016.5,NaN,0.689949,31.150010,44.635441,27.101730,29.439392
136,1501,7.486284,2.142604,4.278479,0.523835,0.720675,1688.992494,433.107657,923.958614,210.824871,...,0.695512,0.658773,NaN,5349.0,NaN,0.620216,47.620275,43.362852,37.554216,35.086378
